In [186]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import f1_score, cohen_kappa_score 

In [187]:
X_train = pd.read_csv("dataset/processed/X_train_featured.csv")
X_test = pd.read_csv("dataset/processed/X_test_featured.csv")
y_train = pd.read_csv("dataset/processed/y_train.csv")
y_test = pd.read_csv("dataset/processed/y_test.csv")

print("\n" + "="*40)
print("DATA SHAPE RESULTS")
print("="*40)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print("-" * 40)
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")


DATA SHAPE RESULTS
X_train shape: (2492096, 37)
y_train shape: (2492096, 1)
----------------------------------------
X_test shape:  (623019, 37)
y_test shape:  (623019, 1)


In [188]:
results_table = []

def evaluate_regressor(model, X_train, y_train, X_test, y_test, model_name):
    # 1. PREDICTION (Predict continuous values)
    y_train_pred_raw = model.predict(X_train)
    y_test_pred_raw = model.predict(X_test)
    
    # 2. PROCESSING (Clip values between 1-4 and round for classification metrics)
    y_train_pred_class = np.round(np.clip(y_train_pred_raw, 1, 4)).astype(int)
    y_test_pred_class = np.round(np.clip(y_test_pred_raw, 1, 4)).astype(int)
    
    # 3. COMPUTE METRICS
    
    train_f1_macro = f1_score(y_train, y_train_pred_class, average='macro')
    test_f1_macro = f1_score(y_test, y_test_pred_class, average='macro')
    
    # F1-score (classification metric)
    train_f1_weighted = f1_score(y_train, y_train_pred_class, average='weighted')
    test_f1_weighted = f1_score(y_test, y_test_pred_class, average='weighted')
    
    # Quadratic Weighted Kappa (ordinal metric)
    train_qwk = cohen_kappa_score(y_train, y_train_pred_class, weights='quadratic')
    test_qwk = cohen_kappa_score(y_test, y_test_pred_class, weights='quadratic')
    
    # 4. PRINT RESULTS
    print(f"✅ {model_name} Results:")
    print(f"   [TRAIN] QWK: {train_qwk:.4f} | F1_macro: {train_f1_macro:.4f} | F1_weight: {train_f1_weighted:.4f}")
    print(f"   [TEST]  QWK: {test_qwk:.4f} | F1_macro: {test_f1_macro:.4f} | F1_weight: {test_f1_weighted:.4f}")

    # 5. CREATE DETAIL DF
    comp_df = pd.DataFrame({
    "y_true": np.array(y_test).ravel(),
    "y_pred": np.array(y_test_pred_class).ravel(),
    "y_raw": np.array(y_test_pred_raw).ravel()
    })

    # 6. PRINT PREVIEW
    print(comp_df[comp_df.y_true != 2].head(10))
   

    print("-" * 70)

    # 7. RETURN RESULT DICTIONARY
    return {
        "Model": model_name,
        "F1_macro": test_f1_macro,
        "F1_weighted": test_f1_weighted,
        "QWK": test_qwk,
        "Object": model
    }


### `evaluate_regressor` Function

This function evaluates a regression model for an **ordinal classification task** (severity 1–4) using the most relevant metrics: **MAE, F1-score (weighted), and Quadratic Weighted Kappa (QWK)**.

## Steps

1. **Prediction**  
   Generate raw predictions on training and test datasets.

2. **Processing**  
   Clip predictions to the range `[1, 4]` and round them to compute classification-like metrics.

3. **Metrics Computation**  
   - **MAE (Mean Absolute Error):** Measures the average deviation of predictions from true values in severity levels.  
   - **F1-score (weighted):** Measures the balance between precision and recall, accounting for class imbalance.  
   - **Quadratic Weighted Kappa (QWK):** Evaluates ordinal agreement, penalizing predictions more when they are farther from the true severity.

4. **Results Display**  
   Print training and test metrics, including the F1 gap to indicate potential overfitting.

5. **Return Value**  
   Return a dictionary containing test metrics (`MAE`, `F1`, `QWK`) and the fitted model object for comparison or saving.

In [189]:
print("🌲 Running Random Forest Regressor")
start = time.time()

# 1. Initialize
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

# 2. Train
rf_model.fit(X_train, y_train.values.ravel())

print("\n📊 Calculating Feature Importance...")

# Create DataFrame with feature names and importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
})

# Sort by importance descending
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Print Top 10 most important features
print("\n🏆 Top 10 Features affecting Severity:")
print(importance_df.head(10).to_string(index=False))

# 3. Evaluate
results_table.append(evaluate_regressor(rf_model, X_train, y_train.values.ravel(), X_test, y_test.values.ravel(), "Random Forest"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🌲 Running Random Forest Regressor

📊 Calculating Feature Importance...

🏆 Top 10 Features affecting Severity:
         Feature  Importance
    City_encoded    0.580094
  Accidents_City    0.098499
  County_encoded    0.064524
   State_encoded    0.038454
       Month_Cos    0.030783
           Month    0.030459
  Traffic_Signal    0.022911
       Month_Sin    0.022826
        Is_Clear    0.021246
Accidents_County    0.016254
✅ Random Forest Results:
   [TRAIN] QWK: 0.2154 | F1_macro: 0.3834 | F1_weight: 0.9087
   [TEST]  QWK: 0.2138 | F1_macro: 0.3841 | F1_weight: 0.9040
     y_true  y_pred     y_raw
34        3       2  2.088533
43        4       3  2.999238
51        4       2  2.006178
83        4       4  3.721032
101       4       2  2.422887
111       1       2  2.019513
136       4       2  2.470494
147       4       2  2.113900
155       4       2  2.023461
177       4       2  2.320984
----------------------------------------------------------------------
⏱️ Time: 594.25s


In [190]:
weights_dict = {1: 5, 2: 3, 3: 5, 4: 10} 
sample_weights = y_train.squeeze().map(weights_dict)

In [191]:
print("🚀 Running XGBoost Regressor")
start = time.time()

# 1. Initialize with fixed parameters
xgb_model = XGBRegressor( 
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=42, 
    n_jobs=-1, 
    verbosity=0 
)

# 2. Train
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

print("\n📊 Calculating Feature Importance (XGBoost)...")

# Create DataFrame with feature names and importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
})

# Sort by importance descending
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Print Top 10 most important features
print("\n🏆 Top 10 Features affecting Severity:")
print(importance_df.head(10).to_string(index=False))

# 3. Evaluate
results_table.append(evaluate_regressor(xgb_model, X_train, y_train, X_test, y_test, "XGBoost"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🚀 Running XGBoost Regressor

📊 Calculating Feature Importance (XGBoost)...

🏆 Top 10 Features affecting Severity:
       Feature  Importance
  City_encoded    0.345193
County_encoded    0.118082
Accidents_City    0.052857
      Is_Clear    0.048929
Traffic_Signal    0.039792
 State_encoded    0.035664
         Month    0.035146
      Hour_Cos    0.034273
      Junction    0.030441
     Month_Cos    0.030290
✅ XGBoost Results:
   [TRAIN] QWK: 0.3135 | F1_macro: 0.3601 | F1_weight: 0.8805
   [TEST]  QWK: 0.2894 | F1_macro: 0.3692 | F1_weight: 0.8727
     y_true  y_pred     y_raw
34        3       2  2.261189
43        4       3  2.992042
51        4       2  2.027802
83        4       4  3.753523
101       4       3  3.002803
111       1       2  2.035281
136       4       3  3.077829
147       4       2  2.357488
155       4       2  2.086763
177       4       3  2.847655
----------------------------------------------------------------------
⏱️ Time: 24.12s


In [192]:
print("⚡ Running LightGBM Regressor")
start = time.time()

# 1. Initialize
lgbm_model = LGBMRegressor( 
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=42, 
    n_jobs=-1, 
    verbose=-1 
)

# 2. Train
lgbm_model.fit(X_train, y_train, sample_weight=sample_weights)

print("\n📊 Calculating Feature Importance (LightGBM)...")

# Create DataFrame with feature names and importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': lgbm_model.feature_importances_
})

# Sort by importance descending
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Print Top 10 most important features
print("\n🏆 Top 10 Features affecting Severity:")
print(importance_df.head(10).to_string(index=False))

# 3. Evaluate
results_table.append(evaluate_regressor(lgbm_model, X_train, y_train, X_test, y_test, "LightGBM"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

⚡ Running LightGBM Regressor

📊 Calculating Feature Importance (LightGBM)...

🏆 Top 10 Features affecting Severity:
         Feature  Importance
    City_encoded         404
   State_encoded         386
  County_encoded         345
  Accidents_City         264
Accidents_County         186
           Month         164
       Month_Cos         141
  Temperature(F)         136
            Hour          97
        Hour_Cos          96
✅ LightGBM Results:
   [TRAIN] QWK: 0.3140 | F1_macro: 0.3616 | F1_weight: 0.8807
   [TEST]  QWK: 0.2871 | F1_macro: 0.3708 | F1_weight: 0.8727
     y_true  y_pred     y_raw
34        3       2  2.268793
43        4       3  3.048071
51        4       2  2.022126
83        4       4  3.823123
101       4       3  2.975836
111       1       2  2.031157
136       4       3  3.043403
147       4       2  2.350304
155       4       2  2.100316
177       4       3  2.914583
----------------------------------------------------------------------
⏱️ Time: 22.89s


In [193]:
print("🐱 Running CatBoost Regressor")
start = time.time()

# 1. Initialize
cat_model = CatBoostRegressor(
    n_estimators=50, 
    random_state=42, 
    verbose=0
)

# 2. Train
cat_model.fit(X_train, y_train, sample_weight=sample_weights)

print("\n📊 Calculating Feature Importance (CatBoost)...")

# Create DataFrame with feature names and importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': cat_model.feature_importances_
})

# Sort by importance descending
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Print Top 10 most important features
print("\n🏆 Top 10 Features affecting Severity:")
print(importance_df.head(10).to_string(index=False))

# 3. Evaluate
results_table.append(evaluate_regressor(cat_model, X_train, y_train, X_test, y_test, "CatBoost"))

print(f"⏱️ Time: {time.time()-start:.2f}s")

🐱 Running CatBoost Regressor

📊 Calculating Feature Importance (CatBoost)...

🏆 Top 10 Features affecting Severity:
         Feature  Importance
    City_encoded   33.235641
  County_encoded   15.415621
  Accidents_City   11.817295
   State_encoded    8.510268
       Month_Sin    4.425344
       Month_Cos    3.465199
        Hour_Cos    3.364867
        Is_Clear    2.996136
  Traffic_Signal    2.964138
Accidents_County    2.940848
✅ CatBoost Results:
   [TRAIN] QWK: 0.3153 | F1_macro: 0.3728 | F1_weight: 0.8804
   [TEST]  QWK: 0.2851 | F1_macro: 0.3585 | F1_weight: 0.8709
     y_true  y_pred     y_raw
34        3       2  2.269275
43        4       3  3.105387
51        4       2  2.052394
83        4       4  4.138267
101       4       3  3.121540
111       1       2  2.008084
136       4       3  3.065157
147       4       2  2.283458
155       4       2  2.075961
177       4       3  3.107896
----------------------------------------------------------------------
⏱️ Time: 18.61s


In [194]:
# 1. Create comparison DataFrame
df_results = pd.DataFrame(results_table)

# Sort by QWK (highest first) — prioritize ordinal agreement
df_results = df_results.sort_values(by='QWK', ascending=False)

# Columns to display
display_cols = ['Model', 'QWK', 'F1_macro', 'F1_weighted']

print("\n🏆 MODEL LEADERBOARD")
print(df_results[display_cols].to_markdown(index=False))

# 2. Select the best model based on QWK
best_model_info = df_results.iloc[0]
best_model_name = best_model_info['Model']
best_model_object = best_model_info['Object']

print(f"\n🥇 WINNER: {best_model_name}")
print(f"   with QWK: {best_model_info['QWK']:.4f}, F1_macro: {best_model_info['F1_macro']:.4f}, F1_weighted: {best_model_info['F1_weighted']:.4f}")

# 3. Save the best model
filename = f"best_model_{best_model_name.replace(' ', '_')}.pkl"
joblib.dump(best_model_object, filename)
print(f"💾 Saved the best model to file: {filename}")


🏆 MODEL LEADERBOARD
| Model         |      QWK |   F1_macro |   F1_weighted |
|:--------------|---------:|-----------:|--------------:|
| XGBoost       | 0.2894   |   0.369219 |      0.872701 |
| LightGBM      | 0.287131 |   0.370849 |      0.872708 |
| CatBoost      | 0.285064 |   0.358535 |      0.870925 |
| Random Forest | 0.213754 |   0.384109 |      0.90396  |

🥇 WINNER: XGBoost
   with QWK: 0.2894, F1_macro: 0.3692, F1_weighted: 0.8727
💾 Saved the best model to file: best_model_XGBoost.pkl


# 📊 COMPARATIVE PERFORMANCE

### 1. The Transformation Matrix

| Model | Metric | Before Processing | After Processing | Delta / Growth | Insight |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **XGBoost** | **QWK** | 0.0890 | **0.2894** | **+225%** 🚀 | From "underdog" to **Champion**. |
| **LightGBM** | QWK | 0.0777 | 0.2871 | **+269%** | Strongest growth rate. |
| **CatBoost** | QWK | 0.1205 | 0.2851 | +136% | Massive improvement. |
| **Random Forest** | QWK | 0.1633 | 0.2138 | +31% | Improved, but fell behind the Boosting family. |
| **XGBoost** | F1-Weighted | **0.9105** | 0.8727 | -4.1% | **Strategic Trade-off** (Less artificial accuracy, more real value). |

---

### 2. Detailed Analysis of Improvements

#### **A. The Quantum Leap in QWK (True Quality)**
This is the most undeniable proof that clean data has fundamentally changed the model's "thinking":
* **Before:** Boosting models (XGBoost, LightGBM) performed terribly (QWK < 0.1). They were highly sensitive to noise (outliers) and imbalance, essentially guessing randomly or making catastrophic errors (e.g., predicting a fatal accident as a minor fender bender).
* **After:** QWK tripled (reaching ~0.29). This proves the models now **understand the ordinal nature** of the data (Severity 1 < 2 < 3 < 4). They don't just predict the correct class; when they make a mistake, it is a "logical" mistake (e.g., predicting 3 instead of 4, rather than 1).

#### **B. Deciphering the F1-Weighted Decline (Honesty over Hype)**
* **Before (F1 ~ 0.91):** This number was a **statistical illusion**. Since Severity 2 makes up the vast majority (~70%), a "lazy" model could simply predict "Severity 2" for everything and still achieve a high score. However, it missed 100% of severe accidents.
* **After (F1 ~ 0.87):** The score dropped slightly because the model started **daring to predict difficult classes (Severity 1, 3, 4)**. By accepting the risk of prediction, the overall error rate increased slightly, but in exchange, we gained the critical ability to detect high-risk events (which the old model completely failed to do).

#### **C. The "Leaderboard Flip"**
* **Before:** **Random Forest** led the pack. Because Bagging algorithms are robust to noise and tend to vote with the majority, Random Forest won easily on the "dirty" data.
* **After:** **XGBoost / LightGBM / CatBoost** dominated the Top 3. Once the data was cleaned, scaled, and outliers capped, these sophisticated algorithms finally had a quality environment to leverage their deep learning capabilities, far outperforming the simpler Random Forest.

---

### 🎯 CONCLUSION

> *"Raw data is like looking through a foggy windshield: Random Forest could vaguely guess the shapes, but precision instruments like XGBoost were completely blinded.
>
> The **Data Preparation** process was the act of wiping that windshield clean. The result is that **XGBoost** can now see clearly, surpassing Random Forest to become the superior model, with an ability to recognize accident severity that is **3 times better** than the initial baseline."*